# BÀI THỰC HÀNH 2: MẠNG NEURAL TÍCH CHẬP
Một notebook cho cả bốn câu của `assignments.ipynb`. Phần chuẩn bị, dataset và hàm tính metric ở đầu được dùng chung. **Mỗi mô hình có ba ô riêng: Train → Test → Kết quả**.

## 0. Chuẩn bị chung

Cần `torch`, `torchvision`, `numpy`, `scikit-learn`, `Pillow`; câu 4 cần `transformers`. Tensor ảnh PyTorch có dạng `[N,C,H,W]`.

In [1]:
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import (accuracy_score, classification_report,
                             precision_recall_fscore_support)
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SOURCE_ROOT = Path("VinaFood21")
CACHE_ROOT = Path("VinaFood21_cache320")
SEED = 42
EPOCHS = 20
TRAIN_MODELS = {"lenet", "googlenet", "resnet18", "resnet50_hf"} # Ví dụ: {"lenet", "googlenet", "resnet18", "resnet50_hf"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(4)
print("device:", DEVICE, "source:", SOURCE_ROOT.resolve())
print("Bật Train/Test riêng ở phần cấu hình của từng mô hình.")

device: cuda source: D:\UIT\DS\DS201\BaiTH02\VinaFood21
Bật Train/Test riêng ở phần cấu hình của từng mô hình.


## 1. Dataset chung

Tập gốc: `VinaFood21/train/<lớp>/` và `VinaFood21/test/<lớp>/`. Chỉ số train/validation được tạo **một lần** bằng seed 42 và dùng cho cả bốn mô hình. Câu 2–4 dùng cache ảnh cạnh dài tối đa 320 pixel để đọc nhanh hơn; cache chỉ phục vụ nạp dữ liệu, không phải kết quả mô hình.

In [2]:
def prepare_cache():
    marker = CACHE_ROOT / ".complete"
    if marker.is_file():
        print("Dùng cache ảnh:", CACHE_ROOT)
        return
    jobs = []
    for split in ("train", "test"):
        ds = datasets.ImageFolder(SOURCE_ROOT / split)
        for i, (src, class_id) in enumerate(ds.samples):
            dst = CACHE_ROOT / split / ds.classes[class_id] / f"{i:06d}.jpg"
            jobs.append((src, dst))

    def convert(job):
        src, dst = job
        if dst.is_file():
            return
        dst.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(src) as image:
            image.draft("RGB", (320, 320))
            image = image.convert("RGB")
            image.thumbnail((320, 320), Image.Resampling.BILINEAR)
            image.save(dst, "JPEG", quality=85, optimize=False)

    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, _ in enumerate(pool.map(convert, jobs), 1):
            if i % 2000 == 0 or i == len(jobs):
                print(f"Đã xử lý {i}/{len(jobs)} ảnh", flush=True)
    marker.write_text(f"{len(jobs)} images\n", encoding="utf-8")
    print("Cache sẵn sàng:", CACHE_ROOT)

prepare_cache()
raw_train = datasets.ImageFolder(SOURCE_ROOT / "train")
raw_test = datasets.ImageFolder(SOURCE_ROOT / "test")
CLASSES = raw_train.classes
assert CLASSES == raw_test.classes, "Tên/thứ tự lớp train và test không khớp"

# Một lần chia cố định; mọi mô hình dùng cùng chỉ số ảnh.
rng = random.Random(SEED)
TRAIN_IDX, VAL_IDX = [], []
for class_id in range(len(CLASSES)):
    indices = [i for i, target in enumerate(raw_train.targets) if target == class_id]
    assert len(indices) >= 2, f"Lớp {CLASSES[class_id]} quá ít ảnh"
    rng.shuffle(indices)
    n_val = max(1, round(len(indices) * 0.1))
    VAL_IDX.extend(indices[:n_val])
    TRAIN_IDX.extend(indices[n_val:])
print("classes:", len(CLASSES), "train:", len(TRAIN_IDX),
      "validation:", len(VAL_IDX), "test:", len(raw_test))
print(CLASSES)

Dùng cache ảnh: VinaFood21_cache320
classes: 21 train: 9038 validation: 1006 test: 6682
['banh-can', 'banh-hoi', 'banh-mi-chao', 'banh-tet', 'banh-trang-tron', 'banh-u', 'banh-uot', 'bap-nuong', 'bo-kho', 'bo-la-lot', 'bot-chien', 'ca-ri', 'canh-kho-qua', 'canh-khoai-mo', 'ga-nuong', 'goi-ga', 'ha-cao', 'hoanh-thanh-nuoc', 'pha-lau', 'tau-hu', 'thit-kho-trung']


In [3]:
def make_loaders(image_size, batch_size, use_cache):
    root = CACHE_ROOT if use_cache else SOURCE_ROOT

    def rgb_loader(path):
        with Image.open(path) as image:
            image.draft("RGB", (max(image_size * 2, 256), max(image_size * 2, 256)))
            return image.convert("RGB")

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    ])
    full_train = datasets.ImageFolder(root / "train", transform=train_tf, loader=rgb_loader)
    full_eval = datasets.ImageFolder(root / "train", transform=eval_tf, loader=rgb_loader)
    test_ds = datasets.ImageFolder(root / "test", transform=eval_tf, loader=rgb_loader)
    assert full_train.classes == test_ds.classes == CLASSES
    assert full_train.targets == raw_train.targets, "Cache không giữ đúng thứ tự ảnh train"
    assert len(test_ds) == len(raw_test)
    train_ds, val_ds = Subset(full_train, TRAIN_IDX), Subset(full_eval, VAL_IDX)
    args = dict(batch_size=batch_size, num_workers=0, pin_memory=DEVICE.type == "cuda")
    return (DataLoader(train_ds, shuffle=True, **args),
            DataLoader(val_ds, shuffle=False, **args),
            DataLoader(test_ds, shuffle=False, **args))

## 2. Hàm train và test dùng chung

Train dùng `CrossEntropyLoss`, Adam và backpropagation. Validation/test dùng `model.eval()` và không cập nhật trọng số. Checkpoint tốt nhất được giữ **trong bộ nhớ** theo validation macro-F1.

In [4]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None,
              use_amp=False, clip_grad=False):
    training = optimizer is not None
    model.train(training)
    total_loss, count = 0.0, 0
    y_true, y_pred = [], []
    start = time.perf_counter()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp and DEVICE.type == "cuda"):
                logits = model(xb)
                loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise FloatingPointError("Loss không hữu hạn; kiểm tra LR và AMP")
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                if clip_grad:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * len(yb)
        count += len(yb)
        y_true.extend(yb.cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(CLASSES))),
        average="macro", zero_division=0)
    return ({"loss": total_loss / count, "accuracy": accuracy_score(y_true, y_pred),
             "precision_macro": p, "recall_macro": r, "f1_macro": f1,
             "seconds": time.perf_counter() - start}, y_true, y_pred)


def fit_in_memory(name, model_factory, image_size, batch_size, lr, epochs,
                  use_amp=False, use_cache=True, clip_grad=False):
    # Cố định seed lại trước từng mô hình để có thể chạy từng câu độc lập.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    train_dl, val_dl, _ = make_loaders(image_size, batch_size, use_cache)
    model = model_factory(len(CLASSES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and DEVICE.type == "cuda")
    best_f1, best_epoch, best_state, history = -1.0, None, None, []
    for epoch in range(1, epochs + 1):
        train, _, _ = run_epoch(model, train_dl, criterion, optimizer, scaler,
                                use_amp, clip_grad)
        val, _, _ = run_epoch(model, val_dl, criterion, use_amp=use_amp)
        history.append({"epoch": epoch, "train": train, "validation": val})
        print(f"{name} epoch {epoch}/{epochs}: train loss={train['loss']:.4f}, "
              f"val loss={val['loss']:.4f}, val macro-F1={val['f1_macro']:.4f}", flush=True)
        if val["f1_macro"] > best_f1:
            best_f1, best_epoch = val["f1_macro"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    print(f"{name}: đã train đủ {epochs} epoch; epoch tốt nhất = {best_epoch}, "
          f"validation macro-F1 = {best_f1:.4f}")
    return model, history, best_epoch, best_f1


def test_in_memory(model, image_size, batch_size, use_amp=False, use_cache=True):
    if model is None:
        raise RuntimeError("Chưa có model trong bộ nhớ. Hãy chạy ô Train của mô hình này trước.")
    _, _, test_dl = make_loaders(image_size, batch_size, use_cache)
    metrics, y_true, y_pred = run_epoch(model, test_dl, nn.CrossEntropyLoss(),
                                        use_amp=use_amp)
    report = classification_report(y_true, y_pred,
                                   labels=list(range(len(CLASSES))),
                                   target_names=CLASSES, zero_division=0, digits=4)
    return {"metrics": metrics, "report": report}


def show_metrics(metrics):
    print(f"Accuracy        : {metrics['accuracy']:.4f}")
    print(f"Macro-precision : {metrics['precision_macro']:.4f}")
    print(f"Macro-recall    : {metrics['recall_macro']:.4f}")
    print(f"Macro-F1        : {metrics['f1_macro']:.4f}")

## Bài 1 — LeNet-5

Các ô của phần này chỉ tác động đến lenet. Đặt `TRAIN_LENET = True` để train, `TEST_LENET = True` để test model vừa train. Mặc định đều tắt để mở notebook nhanh. Kết quả của lần chạy 3 epoch cũ được ghi trong ô Kết quả để tham khảo.

In [5]:
class LeNet5(nn.Module):
    """LeNet-5 cho ảnh RGB; ảnh mặc định 64 x 64."""

    def __init__(self, num_classes: int, image_size: int = 64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5), nn.ReLU(), nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.AvgPool2d(2),
        )
        with torch.no_grad():
            flat_dim = self.features(torch.zeros(1, 3, image_size, image_size)).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flat_dim, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# Cấu hình riêng cho Bài 1 — LeNet-5
EPOCHS_LENET = 20
TRAIN_LENET = True
TEST_LENET = True
IMAGE_SIZE_LENET = 64
BATCH_SIZE_LENET = 64
LR_LENET = 0.001
lenet_model = None
lenet_history = None
lenet_best_epoch = None
lenet_test_result = None
print("lenet:", EPOCHS_LENET, "epochs; train=", TRAIN_LENET, "test=", TEST_LENET)

lenet: 20 epochs; train= False test= False


In [7]:
# TRAIN riêng: Bài 1 — LeNet-5
if TRAIN_LENET:
    lenet_model, lenet_history, lenet_best_epoch, lenet_best_f1 = fit_in_memory(
        "lenet", lambda n: LeNet5(n, 64), IMAGE_SIZE_LENET, BATCH_SIZE_LENET,
        LR_LENET, EPOCHS_LENET, use_amp=True, use_cache=False, clip_grad=False)
else:
    print("Bỏ qua train lenet. Đặt TRAIN_LENET = True trong ô cấu hình để chạy.")

Bỏ qua train lenet. Đặt TRAIN_LENET = True trong ô cấu hình để chạy.


In [8]:
# TEST riêng: Bài 1 — LeNet-5. Dùng model ở epoch có validation macro-F1 tốt nhất.
if TEST_LENET:
    lenet_test_result = test_in_memory(
        lenet_model, IMAGE_SIZE_LENET, BATCH_SIZE_LENET,
        use_amp=True, use_cache=False)
    print("Đã test lenet trên", len(raw_test), "ảnh.")
else:
    print("Bỏ qua test lenet. Đặt TEST_LENET = True trong ô cấu hình để chạy.")

Bỏ qua test lenet. Đặt TEST_LENET = True trong ô cấu hình để chạy.


In [ ]:
# KẾT QUẢ riêng: Bài 1 — LeNet-5
if lenet_test_result is not None:
    print("Kết quả test của lần chạy hiện tại:")
    print("Epoch tốt nhất:", lenet_best_epoch)
    show_metrics(lenet_test_result["metrics"])
    print("\nBáo cáo từng lớp:\n", lenet_test_result["report"])
else lenet_model is not None:
    print("Đã train lenet; hãy bật TEST_LENET = True và chạy ô Test để có kết quả mới.")

Kết quả tham khảo của lần chạy 3 epoch trước (chưa train/test lại trong phiên này):
Accuracy        : 0.2360
Macro-precision : 0.2068
Macro-recall    : 0.2108
Macro-F1        : 0.1750


## Bài 2 — GoogLeNet

Các ô của phần này chỉ tác động đến googlenet. Đặt `TRAIN_GOOGLENET = True` để train, `TEST_GOOGLENET = True` để test model vừa train. Mặc định đều tắt để mở notebook nhanh. Kết quả của lần chạy 3 epoch cũ được ghi trong ô Kết quả để tham khảo.

In [10]:
class Inception(nn.Module):
    """Bốn nhánh 1x1, 3x3, 5x5, max-pool; nối theo chiều channel."""

    def __init__(self, in_ch, f1, f3r, f3, f5r, f5, fp):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(in_ch, f1, 1), nn.ReLU())
        self.b2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, 1), nn.ReLU(),
            nn.Conv2d(f3r, f3, 3, padding=1), nn.ReLU(),
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch, f5r, 1), nn.ReLU(),
            nn.Conv2d(f5r, f5, 5, padding=2), nn.ReLU(),
        )
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, fp, 1), nn.ReLU(),
        )

    def forward(self, x):
        return torch.cat((self.b1(x), self.b2(x), self.b3(x), self.b4(x)), dim=1)

class GoogLeNet(nn.Module):
    """GoogLeNet v1 với 9 Inception blocks và global average pooling.

    Bản sườn chưa có hai auxiliary classifiers của kiến trúc gốc.
    """

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3), nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64, 64, 1), nn.ReLU(),
            nn.Conv2d(64, 192, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
            Inception(192, 64, 96, 128, 16, 32, 32),   # 3a -> 256
            Inception(256, 128, 128, 192, 32, 96, 64), # 3b -> 480
            nn.MaxPool2d(3, stride=2, padding=1),
            Inception(480, 192, 96, 208, 16, 48, 64), # 4a -> 512
            Inception(512, 160, 112, 224, 24, 64, 64),# 4b -> 512
            Inception(512, 128, 128, 256, 24, 64, 64),# 4c -> 512
            Inception(512, 112, 144, 288, 32, 64, 64),# 4d -> 528
            Inception(528, 256, 160, 320, 32, 128, 128), # 4e -> 832
            nn.MaxPool2d(3, stride=2, padding=1),
            Inception(832, 256, 160, 320, 32, 128, 128), # 5a -> 832
            Inception(832, 384, 192, 384, 48, 128, 128), # 5b -> 1024
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(1024, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# Cấu hình riêng cho Bài 2 — GoogLeNet
EPOCHS_GOOGLENET = 20
TRAIN_GOOGLENET = True
TEST_GOOGLENET = True
IMAGE_SIZE_GOOGLENET = 160
BATCH_SIZE_GOOGLENET = 32
LR_GOOGLENET = 0.0003
googlenet_model = None
googlenet_history = None
googlenet_best_epoch = None
googlenet_test_result = None
print("googlenet:", EPOCHS_GOOGLENET, "epochs; train=", TRAIN_GOOGLENET, "test=", TEST_GOOGLENET)

googlenet: 20 epochs; train= False test= False


In [12]:
# TRAIN riêng: Bài 2 — GoogLeNet
if TRAIN_GOOGLENET:
    googlenet_model, googlenet_history, googlenet_best_epoch, googlenet_best_f1 = fit_in_memory(
        "googlenet", lambda n: GoogLeNet(n), IMAGE_SIZE_GOOGLENET, BATCH_SIZE_GOOGLENET,
        LR_GOOGLENET, EPOCHS_GOOGLENET, use_amp=False, use_cache=True, clip_grad=True)
else:
    print("Bỏ qua train googlenet. Đặt TRAIN_GOOGLENET = True trong ô cấu hình để chạy.")

Bỏ qua train googlenet. Đặt TRAIN_GOOGLENET = True trong ô cấu hình để chạy.


In [13]:
# TEST riêng: Bài 2 — GoogLeNet. Dùng model ở epoch có validation macro-F1 tốt nhất.
if TEST_GOOGLENET:
    googlenet_test_result = test_in_memory(
        googlenet_model, IMAGE_SIZE_GOOGLENET, BATCH_SIZE_GOOGLENET,
        use_amp=False, use_cache=True)
    print("Đã test googlenet trên", len(raw_test), "ảnh.")
else:
    print("Bỏ qua test googlenet. Đặt TEST_GOOGLENET = True trong ô cấu hình để chạy.")

Bỏ qua test googlenet. Đặt TEST_GOOGLENET = True trong ô cấu hình để chạy.


In [ ]:
# KẾT QUẢ riêng: Bài 2 — GoogLeNet
if googlenet_test_result is not None:
    print("Kết quả test của lần chạy hiện tại:")
    print("Epoch tốt nhất:", googlenet_best_epoch)
    show_metrics(googlenet_test_result["metrics"])
    print("\nBáo cáo từng lớp:\n", googlenet_test_result["report"])
else googlenet_model is not None:
    print("Đã train googlenet; hãy bật TEST_GOOGLENET = True và chạy ô Test để có kết quả mới.")

Kết quả tham khảo của lần chạy 3 epoch trước (chưa train/test lại trong phiên này):
Accuracy        : 0.1202
Macro-precision : 0.0454
Macro-recall    : 0.0996
Macro-F1        : 0.0519


## Bài 3* — ResNet-18

Các ô của phần này chỉ tác động đến resnet18. Đặt `TRAIN_RESNET18 = True` để train, `TEST_RESNET18 = True` để test model vừa train. Mặc định đều tắt để mở notebook nhanh. Kết quả của lần chạy 3 epoch cũ được ghi trong ô Kết quả để tham khảo.

In [15]:
class BasicBlock(nn.Module):
    """Residual block: F(x) + shortcut(x)."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.shortcut = nn.Identity() if stride == 1 and in_ch == out_ch else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        return torch.relu(self.main(x) + self.shortcut(x))

class ResNet18(nn.Module):
    """ResNet-18 tự xây dựng theo số block [2, 2, 2, 2]."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(3, 2, 1),
        )
        self.stages = nn.Sequential(
            self._stage(64, 64, 1), self._stage(64, 128, 2),
            self._stage(128, 256, 2), self._stage(256, 512, 2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    @staticmethod
    def _stage(in_ch, out_ch, stride):
        return nn.Sequential(BasicBlock(in_ch, out_ch, stride), BasicBlock(out_ch, out_ch))

    def forward(self, x):
        return self.fc(self.pool(self.stages(self.stem(x))).flatten(1))

In [ ]:
# Cấu hình riêng cho Bài 3* — ResNet-18
EPOCHS_RESNET18 = 20
TRAIN_RESNET18 = True
TEST_RESNET18 = True
IMAGE_SIZE_RESNET18 = 96
BATCH_SIZE_RESNET18 = 64
LR_RESNET18 = 0.0003
resnet18_model = None
resnet18_history = None
resnet18_best_epoch = None
resnet18_test_result = None
print("resnet18:", EPOCHS_RESNET18, "epochs; train=", TRAIN_RESNET18, "test=", TEST_RESNET18)

resnet18: 20 epochs; train= False test= False


In [17]:
# TRAIN riêng: Bài 3* — ResNet-18
if TRAIN_RESNET18:
    resnet18_model, resnet18_history, resnet18_best_epoch, resnet18_best_f1 = fit_in_memory(
        "resnet18", lambda n: ResNet18(n), IMAGE_SIZE_RESNET18, BATCH_SIZE_RESNET18,
        LR_RESNET18, EPOCHS_RESNET18, use_amp=False, use_cache=True, clip_grad=True)
else:
    print("Bỏ qua train resnet18. Đặt TRAIN_RESNET18 = True trong ô cấu hình để chạy.")

Bỏ qua train resnet18. Đặt TRAIN_RESNET18 = True trong ô cấu hình để chạy.


In [18]:
# TEST riêng: Bài 3* — ResNet-18. Dùng model ở epoch có validation macro-F1 tốt nhất.
if TEST_RESNET18:
    resnet18_test_result = test_in_memory(
        resnet18_model, IMAGE_SIZE_RESNET18, BATCH_SIZE_RESNET18,
        use_amp=False, use_cache=True)
    print("Đã test resnet18 trên", len(raw_test), "ảnh.")
else:
    print("Bỏ qua test resnet18. Đặt TEST_RESNET18 = True trong ô cấu hình để chạy.")

Bỏ qua test resnet18. Đặt TEST_RESNET18 = True trong ô cấu hình để chạy.


In [ ]:
# KẾT QUẢ riêng: Bài 3* — ResNet-18
if resnet18_test_result is not None:
    print("Kết quả test của lần chạy hiện tại:")
    print("Epoch tốt nhất:", resnet18_best_epoch)
    show_metrics(resnet18_test_result["metrics"])
    print("\nBáo cáo từng lớp:\n", resnet18_test_result["report"])
else resnet18_model is not None:
    print("Đã train resnet18; hãy bật TEST_RESNET18 = True và chạy ô Test để có kết quả mới.")

Kết quả tham khảo của lần chạy 3 epoch trước (chưa train/test lại trong phiên này):
Accuracy        : 0.3002
Macro-precision : 0.3293
Macro-recall    : 0.2879
Macro-F1        : 0.2535


## Bài 4* — ResNet50 Hugging Face

Các ô của phần này chỉ tác động đến resnet50_hf. Đặt `TRAIN_RESNET50_HF = True` để train, `TEST_RESNET50_HF = True` để test model vừa train. Mặc định đều tắt để mở notebook nhanh. Kết quả của lần chạy 3 epoch cũ được ghi trong ô Kết quả để tham khảo.

In [20]:
class HFResNet50(nn.Module):
    """ResNet50 pretrained từ Hugging Face; thay head thành 21 lớp."""

    def __init__(self, num_classes: int):
        super().__init__()
        from transformers import ResNetForImageClassification

        try:
            self.model = ResNetForImageClassification.from_pretrained(
                "microsoft/resnet-50", num_labels=num_classes,
                ignore_mismatched_sizes=True, local_files_only=True)
        except OSError:
            self.model = ResNetForImageClassification.from_pretrained(
                "microsoft/resnet-50", num_labels=num_classes,
                ignore_mismatched_sizes=True)

    def forward(self, x):
        return self.model(pixel_values=x).logits

In [ ]:
# Cấu hình riêng cho Bài 4* — ResNet50 Hugging Face
EPOCHS_RESNET50_HF = 20
TRAIN_RESNET50_HF = True
TEST_RESNET50_HF = True
IMAGE_SIZE_RESNET50_HF = 160
BATCH_SIZE_RESNET50_HF = 16
LR_RESNET50_HF = 0.0001
resnet50_hf_model = None
resnet50_hf_history = None
resnet50_hf_best_epoch = None
resnet50_hf_test_result = None
print("resnet50_hf:", EPOCHS_RESNET50_HF, "epochs; train=", TRAIN_RESNET50_HF, "test=", TEST_RESNET50_HF)

resnet50_hf: 20 epochs; train= False test= False


In [22]:
# TRAIN riêng: Bài 4* — ResNet50 Hugging Face
if TRAIN_RESNET50_HF:
    resnet50_hf_model, resnet50_hf_history, resnet50_hf_best_epoch, resnet50_hf_best_f1 = fit_in_memory(
        "resnet50_hf", lambda n: HFResNet50(n), IMAGE_SIZE_RESNET50_HF, BATCH_SIZE_RESNET50_HF,
        LR_RESNET50_HF, EPOCHS_RESNET50_HF, use_amp=False, use_cache=True, clip_grad=True)
else:
    print("Bỏ qua train resnet50_hf. Đặt TRAIN_RESNET50_HF = True trong ô cấu hình để chạy.")

Bỏ qua train resnet50_hf. Đặt TRAIN_RESNET50_HF = True trong ô cấu hình để chạy.


In [23]:
# TEST riêng: Bài 4* — ResNet50 Hugging Face. Dùng model ở epoch có validation macro-F1 tốt nhất.
if TEST_RESNET50_HF:
    resnet50_hf_test_result = test_in_memory(
        resnet50_hf_model, IMAGE_SIZE_RESNET50_HF, BATCH_SIZE_RESNET50_HF,
        use_amp=False, use_cache=True)
    print("Đã test resnet50_hf trên", len(raw_test), "ảnh.")
else:
    print("Bỏ qua test resnet50_hf. Đặt TEST_RESNET50_HF = True trong ô cấu hình để chạy.")

Bỏ qua test resnet50_hf. Đặt TEST_RESNET50_HF = True trong ô cấu hình để chạy.


In [ ]:
# KẾT QUẢ riêng: Bài 4* — ResNet50 Hugging Face
if resnet50_hf_test_result is not None:
    print("Kết quả test của lần chạy hiện tại:")
    print("Epoch tốt nhất:", resnet50_hf_best_epoch)
    show_metrics(resnet50_hf_test_result["metrics"])
    print("\nBáo cáo từng lớp:\n", resnet50_hf_test_result["report"])
else resnet50_hf_model is not None:
    print("Đã train resnet50_hf; hãy bật TEST_RESNET50_HF = True và chạy ô Test để có kết quả mới.")

Kết quả tham khảo của lần chạy 3 epoch trước (chưa train/test lại trong phiên này):
Accuracy        : 0.6937
Macro-precision : 0.7021
Macro-recall    : 0.7021
Macro-F1        : 0.6853
